# Paso 7 - Ablation Study

Este notebook explicita el analisis de ablacion del proyecto.

Objetivo: aislar el aporte de cada componente del modelo propuesto:

- senal secuencial,
- metadata de categoria directa,
- jerarquia de categorias,
- combinacion hibrida fija,
- pesos adaptativos,
- tuning de hiperparametros.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_DIR = Path('..').resolve().parent
OUTPUT_DIR = PROJECT_DIR / 'H3' / 'outputs'
FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 160)

## 1. Cargar resultados de variantes

Usamos dos fuentes:

- `step2_metrics_summary_sample_50000.csv`: contiene variantes no tuneadas evaluadas en muestra de 50.000 usuarios.
- `step2_best_model_metrics_full.csv` y `final_metrics_summary_full.csv`: contienen el modelo tuneado y baselines finales sobre todos los usuarios evaluables.

In [ ]:
step2_sample = pd.read_csv(OUTPUT_DIR / 'step2_metrics_summary_sample_50000.csv')
final_full = pd.read_csv(OUTPUT_DIR / 'final_metrics_summary_full.csv')
best_full = pd.read_csv(OUTPUT_DIR / 'step2_best_model_metrics_full.csv')

display(step2_sample)
display(final_full)

## 2. Definir variantes de ablacion

Cada fila indica que componentes estan activos. Esto permite interpretar los resultados como un analisis por componentes, no solo como una lista de modelos.

In [ ]:
component_rows = [
    {
        'model': 'Sequential Transition',
        'sequential': True,
        'direct_category': False,
        'hierarchy': False,
        'fixed_weights': False,
        'adaptive_weights': False,
        'tuned': False,
        'interpretation': 'Solo usa transiciones item-item desde el historial reciente.'
    },
    {
        'model': 'Category Popularity',
        'sequential': False,
        'direct_category': True,
        'hierarchy': False,
        'fixed_weights': False,
        'adaptive_weights': False,
        'tuned': False,
        'interpretation': 'Solo usa popularidad dentro de la ultima categoria conocida.'
    },
    {
        'model': 'Hierarchical Category',
        'sequential': False,
        'direct_category': True,
        'hierarchy': True,
        'fixed_weights': False,
        'adaptive_weights': False,
        'tuned': False,
        'interpretation': 'Usa categoria directa y backoff por ancestros, sin secuencia.'
    },
    {
        'model': 'Fixed Hybrid Seq+Category',
        'sequential': True,
        'direct_category': True,
        'hierarchy': False,
        'fixed_weights': True,
        'adaptive_weights': False,
        'tuned': False,
        'interpretation': 'Combina secuencia y categoria directa con pesos fijos.'
    },
    {
        'model': 'Adaptive Hybrid',
        'sequential': True,
        'direct_category': True,
        'hierarchy': False,
        'fixed_weights': False,
        'adaptive_weights': True,
        'tuned': False,
        'interpretation': 'Combina secuencia y categoria directa con pesos adaptativos.'
    },
    {
        'model': 'Adaptive Hierarchical Hybrid',
        'sequential': True,
        'direct_category': True,
        'hierarchy': True,
        'fixed_weights': False,
        'adaptive_weights': True,
        'tuned': False,
        'interpretation': 'Combina secuencia, categoria y jerarquia con pesos adaptativos.'
    },
    {
        'model': 'Adaptive Hierarchical Hybrid (tuned)',
        'sequential': True,
        'direct_category': True,
        'hierarchy': True,
        'fixed_weights': False,
        'adaptive_weights': True,
        'tuned': True,
        'interpretation': 'Misma arquitectura adaptativa jerarquica, con hiperparametros tuneados.'
    },
]

components = pd.DataFrame(component_rows)
display(components)

## 3. Tabla de ablacion en muestra de 50.000 usuarios

Esta tabla permite comparar las variantes no tuneadas bajo el mismo protocolo de muestra. Es la evidencia mas directa para aislar componentes antes del tuning final.

In [ ]:
ablation_sample = components.merge(step2_sample, on='model', how='inner')
ablation_sample = ablation_sample.sort_values('recall@10', ascending=False)
ablation_sample.to_csv(OUTPUT_DIR / 'step7_ablation_sample_50000.csv', index=False)
display(ablation_sample[[
    'model', 'sequential', 'direct_category', 'hierarchy', 'fixed_weights', 'adaptive_weights',
    'precision@10', 'recall@10', 'ndcg@10', 'novelty@10', 'category_diversity@10', 'catalog_coverage@10',
    'interpretation'
]])

## 4. Tabla de ablacion final con modelo tuneado

La tabla final mezcla resultados full de los baselines principales con el modelo tuneado. Para `Adaptive Hybrid` y `Adaptive Hierarchical Hybrid` no tuneados conservamos la evidencia de muestra; para el resultado final reportamos la corrida full.

In [ ]:
full_models = final_full[final_full['model'].isin([
    'Sequential Transition',
    'Category Popularity',
    'Fixed Hybrid Seq+Category',
    'Adaptive Hierarchical Hybrid (tuned)',
])].copy()

sample_only = step2_sample[step2_sample['model'].isin([
    'Hierarchical Category',
    'Adaptive Hybrid',
    'Adaptive Hierarchical Hybrid',
])].copy()
sample_only['evaluation_scope'] = 'sample_50000'
full_models['evaluation_scope'] = 'full_406020'

ablation_mixed = pd.concat([full_models, sample_only], ignore_index=True)
ablation_mixed = components.merge(ablation_mixed, on='model', how='inner')
ablation_mixed = ablation_mixed.sort_values(['evaluation_scope', 'recall@10'], ascending=[True, False])
ablation_mixed.to_csv(OUTPUT_DIR / 'step7_ablation_mixed_final.csv', index=False)
display(ablation_mixed[[
    'model', 'evaluation_scope', 'sequential', 'direct_category', 'hierarchy', 'fixed_weights',
    'adaptive_weights', 'tuned', 'recall@10', 'ndcg@10', 'novelty@10', 'category_diversity@10',
    'catalog_coverage@10', 'interpretation'
]])

## 5. Grafico de ablacion: Recall@10 y nDCG@10

El objetivo es visualizar como cambia la relevancia al agregar componentes.

In [ ]:
plot_order = [
    'Category Popularity',
    'Hierarchical Category',
    'Sequential Transition',
    'Fixed Hybrid Seq+Category',
    'Adaptive Hybrid',
    'Adaptive Hierarchical Hybrid',
]
plot_df = ablation_sample.set_index('model').loc[plot_order].reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(plot_df))
width = 0.38
ax.bar(x - width/2, plot_df['recall@10'], width, label='Recall@10', color='#2f6f8f')
ax.bar(x + width/2, plot_df['ndcg@10'], width, label='nDCG@10', color='#9b7e46')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['model'], rotation=30, ha='right')
ax.set_title('Ablation study en muestra de 50.000 usuarios')
ax.set_ylabel('Metric value')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'step7_ablation_recall_ndcg_sample.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. Mejoras por componente

Calculamos diferencias entre variantes cercanas para cuantificar el aporte de cada bloque.

In [ ]:
sample_idx = step2_sample.set_index('model')

comparisons = [
    ('Categoria directa vs secuencial', 'Category Popularity', 'Sequential Transition'),
    ('Jerarquia sobre categoria directa', 'Category Popularity', 'Hierarchical Category'),
    ('Hibrido fijo sobre secuencial', 'Sequential Transition', 'Fixed Hybrid Seq+Category'),
    ('Adaptacion sobre hibrido fijo', 'Fixed Hybrid Seq+Category', 'Adaptive Hybrid'),
    ('Jerarquia sobre adaptativo', 'Adaptive Hybrid', 'Adaptive Hierarchical Hybrid'),
]

delta_rows = []
for label, base, variant in comparisons:
    for metric in ['recall@10', 'ndcg@10', 'novelty@10', 'category_diversity@10', 'catalog_coverage@10']:
        base_value = sample_idx.loc[base, metric]
        variant_value = sample_idx.loc[variant, metric]
        delta_rows.append({
            'comparison': label,
            'base_model': base,
            'variant_model': variant,
            'metric': metric,
            'base_value': base_value,
            'variant_value': variant_value,
            'absolute_delta': variant_value - base_value,
            'relative_delta': (variant_value - base_value) / base_value if base_value != 0 else np.nan,
        })

component_deltas = pd.DataFrame(delta_rows)
component_deltas.to_csv(OUTPUT_DIR / 'step7_component_deltas_sample_50000.csv', index=False)
display(component_deltas)

## 7. Lectura del ablation study

Conclusiones principales:

- La senal secuencial es mas fuerte que la categoria directa por si sola.
- La categoria directa contiene informacion util, pero no basta para superar a la senal secuencial.
- El hibrido fijo mejora a los modelos individuales, lo que indica complementariedad entre secuencia y categoria.
- Los pesos adaptativos mejoran al hibrido fijo, especialmente en Recall@10.
- La jerarquia agrega una mejora adicional pequena en Recall@10 sobre el hibrido adaptativo sin jerarquia.
- El tuning final entrega la mejor version del modelo propuesto en la evaluacion full.